# Personalized Product Recommender

Objective:
- To recommend relevant and similar products by leveraging product attributes and recent user interaction context, enabling personalized product discovery and seamless navigation across large product catalogs.

Business Use Cases
- Session-based product recommendation: Recommend the next best products based on items recently viewed by a user within a browsing session.

ML Framing:
- Type: Unsupervised Learning (Content-Based Recommendation)
- Target Output: Ranked List of recommended products based on similarity to recent user interations

### Architecture Overview

1. Product attribute preprocessing and feature enrichment using Spark
2. Semantic embedding generation for product text (title + description)
3. **Embedding persistence** in a vector index (e.g., ChromaDB) for reuse across runs
4. **Candidate retrieval (core)** via nearest-neighbor search in embedding space (Top-K)
5. Optional re-ranking, post-filtering, and diversity controls (e.g., cluster-aware constraints)
6. Batch/online recommendation generation (session-based)

### Preprocessing

1. Import libraries
2. Load OpenAI API key
3. Set and create Spark session
4. Load dataset

1. Import libraries

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.clustering import KMeans
import plotly.express as px

os.chdir(r'C:\Users\ashle\Project\usecase')

2. Load OpenAI API Key

In [2]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var. Set it and restart kernel."

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

3. Set and create Spark session

In [3]:
# Set Spark to use same Python as venv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .master("local[*]")
         .appName("ProductRecommender")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.ui.enabled", "false")
         .getOrCreate())



In [4]:
spark = SparkSession.builder.appName("ProductRecommender").getOrCreate()
spark

4. Load dataset

In [5]:
file_path = 'data/products_dataset.csv'
df = spark.read.csv(file_path, header=True, inferSchema=True, samplingRatio=1)
print(df.count())
df.show()

2000
+----------+--------------------+--------------------+
|product_id|               title|         description|
+----------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|
|        P1|Turmode 30 ft. RP...|If you need more ...|
|        P2|Large Tapestry Bo...|Polyester cover r...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|
|        P9|Traditional Silve...|This transitional...|
|       P10|15 in. x 59 in. O...|Its easy to add a...|
|       P11|1 qt. #350F-7 Wil...|BEHR PREMIUM PLUS...|
|       P12|Anthracite Cordle...|BlindsAvenue ligh...|
|       P13|SlimGrip 78-Inch ...|Luverne SlimGrip ...|
|       P14|6 in. x 28 in. x ...|Our Rustic Collec...|
|    

### Feature Engineering

 1. Text combination
 2. Embedding generation
 3. Feature assembly

 1. Text combination

 - Combine title and description column

In [6]:
df = df.withColumn('combined_text', concat_ws(" ", df.title,df.description))
df.show()

+----------+--------------------+--------------------+--------------------+
|product_id|               title|         description|       combined_text|
+----------+--------------------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|
|        P2|Large Tapestry Bo...|Polyester cover r...|Large Tapestry Bo...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|16-Gauge-Sinks Ve...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|Men's Crazy Horse...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|Mariana 6 ft. Mul...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|5 gal. #650C-2 Po...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|7/8 in. x 4-1/2 i...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|Ring Gold Bar Car...|
|        P9|Traditional Silve...|This transitional...|Traditional Silve...|
|       P10|

- Convert the combined_text into a list for embedding

In [7]:
list_combined_text = df.select('combined_text').rdd.flatMap(lambda x: x).collect()
print(list_combined_text[:2])

["Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt This heavyweight, water-repellent hooded sweatshirt has a zip front for fast layering. ORIGINAL FIT. 13 oz., 75% cotton/25% polyester blend with Rain Defender durable water repellent. Attached, jersey-lined three-piece hood with drawcord closure. Antique-finish brass front zipper. Two front hand-warmer pockets have a hidden security pocket inside. Stretchable, spandex-reinforced rib-knit cuffs and waistband. Locker loop facilitates hanging.", "Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable If you need more length between your existing wireless device and Hi-Gain Antenna, this is the product for you. It's compatible with most Wi-Fi Antennas, so it is easy for you to extend your wireless network. Just replace your existing cable that runs between your wireless device and Antenna and you're ready to use your network with extended range."]


 2. Embedding generation

 - Use OpenAI text embedding model to create the vector embeddings

In [8]:
response = client.embeddings.create(
    input=list_combined_text,
    model="text-embedding-3-small",
    dimensions=512
)

- Display first two embedding vectors

In [9]:
embedding_vectors = [data.embedding for data in response.data]
embedding_vectors[:2]  # show first 2 embedding vectors

[[0.04267967492341995,
  0.02093353308737278,
  -0.013637710362672806,
  -0.002072375500574708,
  0.0031974425073713064,
  -0.03727405145764351,
  0.027204759418964386,
  0.07829317450523376,
  0.054974813014268875,
  -0.0628182590007782,
  0.0441989004611969,
  0.04829728230834007,
  -0.0678352415561676,
  0.025226231664419174,
  0.022541087120771408,
  0.0639488473534584,
  0.10104624927043915,
  -0.030843837186694145,
  -0.0889630988240242,
  0.07066170871257782,
  -0.05801326408982277,
  0.06719928979873657,
  -0.034818559885025024,
  -0.07377082854509354,
  0.03450058028101921,
  0.04328029975295067,
  -0.0525369830429554,
  0.025014245882630348,
  0.05112374946475029,
  -0.017250290140509605,
  -0.0031400297302752733,
  -0.022081784904003143,
  0.019131658598780632,
  -0.02457261085510254,
  0.04497617855668068,
  -0.06666932255029678,
  -0.021110186353325844,
  0.10450867563486099,
  -0.038687288761138916,
  0.02563253603875637,
  0.02345968782901764,
  -0.052748966962099075,
  

### Modeling (Retrieval + Optional Segmentation)

This recommender is **content-based**: it recommends products whose embeddings are closest to the user’s **session context** (recently viewed items).

Pipeline (portfolio framing):
1. **Candidate retrieval (core)**: nearest-neighbor search in embedding space (Vector DB / ANN), excluding already-viewed items.
2. **Optional clustering (K-Means)**: use clusters for catalog segmentation and diversity controls (e.g., limit recommendations per cluster) or as a cheap candidate pre-filter.
3. **Dimensionality reduction (PCA)**: visualization/debugging only (not used for ranking).
4. **Recommendation demo**: generate recommendations for a sample “recently viewed” session.

#### Recommendation Engine (Vector DB Retrieval)

Next we switch from “analysis” to the actual recommender path:
- Take a user’s `recently_viewed_products`
- Build a session embedding
- Retrieve the nearest products from a persistent vector index (Chroma)

This is the core of an embedding-based recommender; clustering + PCA remain optional diagnostics.

In [ ]:
# Example session: products the user recently viewed
recently_viewed_products = ["P316", "P333", "P1115", "P1054"]
recently_viewed_products

#### Step 1 — Imports and persistent index location

We use Chroma in persistent mode so the product index is stored in `chroma_store/` and can be reused across runs.

In [ ]:
from pathlib import Path
import numpy as np

import chromadb

PERSIST_DIR = Path("chroma_store") / "products_recommender"
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(PERSIST_DIR))

COLLECTION_NAME = "products_text_embedding_3_small_512"
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print("Persist dir:", PERSIST_DIR)
print("Collection:", COLLECTION_NAME)
print("Current collection count:", collection.count())

#### Step 2 — Prepare product rows for indexing

We index:
- `ids`: your `product_id`
- `documents`: `combined_text` (useful for debugging)
- `metadatas`: lightweight fields like `title`

We also check that the number of embeddings matches the number of products.

In [ ]:
products_pd = df.select("product_id", "title", "combined_text").toPandas()

product_ids = products_pd["product_id"].astype(str).tolist()
titles = products_pd["title"].astype(str).fillna("").tolist()
documents = products_pd["combined_text"].astype(str).fillna("").tolist()

assert len(product_ids) == len(embedding_vectors), (
    f"Mismatch: products={len(product_ids)} vs embeddings={len(embedding_vectors)}"
 )

print("Products:", len(product_ids))
print("Embeddings:", len(embedding_vectors))
print("First product:", product_ids[0], titles[0])

#### Step 3 — Upsert embeddings into Chroma (index build)

If the collection is empty, we insert the vectors. If it already matches the current catalog size, we reuse it.

In [ ]:
expected_count = len(product_ids)
existing_count = collection.count()

print("Existing in collection:", existing_count)
print("Expected products:", expected_count)

if existing_count == 0:
    batch_size = 500
    for start in range(0, expected_count, batch_size):
        end = min(start + batch_size, expected_count)
        metas = [
            {"product_id": product_ids[i], "title": titles[i]}
            for i in range(start, end)
        ]
        collection.upsert(
            ids=product_ids[start:end],
            embeddings=embedding_vectors[start:end],
            documents=documents[start:end],
            metadatas=metas,
        )
    print("Upsert complete.")
elif existing_count == expected_count:
    print("Index already built for this catalog. Reusing.")
else:
    print(
        "Warning: index size differs from catalog size. "
        "For a clean demo, delete persist dir and re-run: ",
        str(PERSIST_DIR),
    )

print("Collection count after:", collection.count())

#### Step 4 — Build a session embedding

We represent the user’s current intent with a simple mean of the embeddings of the recently viewed products.

In [ ]:
id_to_embedding = {pid: emb for pid, emb in zip(product_ids, embedding_vectors)}

viewed = [str(x) for x in recently_viewed_products]
viewed_found = [v for v in viewed if v in id_to_embedding]

print("Viewed:", viewed)
print("Found in catalog:", viewed_found)

if not viewed_found:
    raise ValueError("None of the recently viewed product_ids exist in the indexed catalog")

session_vec = np.asarray([id_to_embedding[v] for v in viewed_found], dtype=float).mean(axis=0).tolist()
print("Session embedding dim:", len(session_vec))

#### Step 5 — Retrieve nearest neighbors from the vector DB

We query Chroma using `session_vec`. We overfetch results so we can remove already-viewed products.

In [ ]:
k = 10
overfetch = 50

raw = collection.query(
    query_embeddings=[session_vec],
    n_results=max(k, overfetch),
    include=["metadatas", "distances"],
 )

print("Returned candidates:", len(raw["ids"][0]))
print("Example raw result:", raw["ids"][0][0], raw["metadatas"][0][0], raw["distances"][0][0])

#### Step 6 — Filter out already viewed items and show top-$k$

Chroma returns a distance score (cosine space). Smaller distance = more similar.

We remove items the user already viewed and take the first $k$ results.

In [ ]:
seen = set(viewed)

recs = []
for pid, meta, dist in zip(raw["ids"][0], raw["metadatas"][0], raw["distances"][0]):
    if pid in seen:
        continue
    recs.append({"product_id": pid, "title": meta.get("title", ""), "distance": float(dist)})
    if len(recs) >= k:
        break

recs

- Convert embedding vectors list into a Pyspark DataFrame

In [ ]:
features_column_names = [f"embedding_{i}" for i in range(len(embedding_vectors[0]))]
embedding_df = spark.createDataFrame(embedding_vectors, schema=features_column_names)
print(embedding_df.count())
embedding_df.show()

Adding unique row_id to each row in embedding_df

In [ ]:
embedding_df= embedding_df.repartition(1).withColumn("id", F.monotonically_increasing_id())
embedding_df.show(2)

Adding unique row_id to each row in main df

In [ ]:
df = df.repartition(1).withColumn("id", F.monotonically_increasing_id())
df.show(2)

Join embedding_df into main df

In [ ]:
df = df.join(embedding_df, on="id", how="inner").drop("id")
df.show(2)

In [ ]:
df.count()

 3. Feature assembly

 - Assemble the 512 embedding columns into a single 'embeddings' column
 - VectorAssembler is used to combine features that are going to be used in the machine learning model

In [ ]:
assembler = VectorAssembler(
    inputCols=features_column_names,
    outputCol="embeddings"
)
data = assembler.transform(df)
data = data.select("product_id", "title", "description", "embeddings")
print(data.count())
data.show(2)

1. (Optional) Clustering for Segmentation

- Apply K-Means clustering (here: `k=5`) on the product embedding vectors.
- Purpose in a recommender:
  - **Segmentation / interpretability**: understand the catalog structure.
  - **Diversity constraints**: avoid recommending 10 near-identical items.
  - **Candidate pre-filter** (optional): retrieve within the user’s cluster(s), then **re-rank by similarity**.

Note: A production-style embedding recommender does *not* require K-Means; you can retrieve directly via vector similarity search.

In [16]:
kmeans = KMeans(k=5, featuresCol='embeddings', predictionCol='cluster')
model = kmeans.fit(data)
clustered_data = model.transform(data)
print(clustered_data.count())
clustered_data.show(5)

2000
+----------+--------------------+--------------------+--------------------+-------+
|product_id|               title|         description|          embeddings|cluster|
+----------+--------------------+--------------------+--------------------+-------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04267967492341...|      4|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|
+----------+--------------------+--------------------+--------------------+-------+
only showing top 5 rows


2. Dimensionality Reduction (Visualization Only)

- Reduce embeddings from 512 dimensions to 2 dimensions for plotting and debugging.
- PCA is used **only** to visualize structure and clusters; recommendations should be based on the full embedding space (or ANN over it).

In [17]:
pca = PCA(k=2, inputCol='embeddings', outputCol='pca_embeddings')
pca_model = pca.fit(clustered_data)
pca_results = pca_model.transform(clustered_data)
pca_results.show(5)

+----------+--------------------+--------------------+--------------------+-------+--------------------+
|product_id|               title|         description|          embeddings|cluster|      pca_embeddings|
+----------+--------------------+--------------------+--------------------+-------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04267967492341...|      4|[0.18867785148774...|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|[-0.1739559627059...|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|[-0.0202184231370...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|[0.00737630557938...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|[-0.0240373666866...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|[0.06058844178915...|      1|[-0.0014908469519...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|[

Convert to pandas dataframe

In [18]:
pca_df = pca_results.select("product_id", "pca_embeddings", "cluster").toPandas()
pca_df

,product_id,pca_embeddings,cluster
0,P0,"[0.18867785148774485, 0.034228364365063484]",4
1,P1,"[-0.17395596270599445, -0.13300941332495123]",4
2,P2,"[-0.020218423137017858, 0.31632337758792767]",2
3,P3,"[0.007376305579386828, 0.0593310913334424]",0
4,P4,"[-0.024037366686663137, -0.04418865487149011]",4
...,...,...,...
1995,P1995,"[0.1509553312518968, 0.29626086048325495]",1
1996,P1996,"[-0.056687769388258105, 0.5848000714343616]",2
1997,P1997,"[0.10783324755670416, -0.0047508389158269874]",1
1998,P1998,"[0.6915404361093128, 0.09694451341111456]",3


3. Visualization

Prepare the data into x and y components

In [19]:
pca_df[['x', 'y']] = pd.DataFrame(pca_df['pca_embeddings'].tolist(), index=pca_df.index)

In [20]:
def plot_clusters(pca_df, num_clusters=5):
    # Create the base cluster plot
    fig = px.scatter(
        pca_df,
        x='x',
        y='y',
        opacity=0.6,
        size_max=4,
        color= pca_df.cluster.astype(str),
        title='2D Visualization of Clusters with Recently Viewed Products',
        labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
        category_orders={'cluster': list(range(num_clusters))},
        # show the product id in the tooltip
        hover_data={'product_id': True}

    )

    # Update layout to add legend title and adjust plot settings
    fig.update_layout(legend_title_text='Clusters', legend=dict(x=1, y=1), width=600, height=500)

    return fig

fig = plot_clusters(pca_df)
fig.show()

### Baseline Demo (Cluster-Filtered Candidates)

Below is your original baseline approach:
- find the cluster(s) containing the recently viewed products
- sample candidates from those cluster(s)

It’s useful for segmentation/diversity, but the vector DB retrieval above is the direct way to produce ranked recommendations.

In [22]:
filtered_data = clustered_data.where(F.col("product_id").isin(recently_viewed_products))
filtered_data.show()

+----------+--------------------+--------------------+--------------------+-------+
|product_id|               title|         description|          embeddings|cluster|
+----------+--------------------+--------------------+--------------------+-------+
|      P316|Mystic Fitz Roy B...|With its distress...|[-0.0157568920403...|      2|
|      P333|Florida Shag Beig...|Lavish natural mo...|[-0.0112434905022...|      2|
|     P1054|1 gal. #HDPG60 Mi...|The improved PPG ...|[-0.0048829163424...|      3|
|     P1115|Modern Gray/Multi...|This Modern Gray/...|[-0.0226364415138...|      2|
+----------+--------------------+--------------------+--------------------+-------+



In [23]:
filtered_data.select("title").rdd.flatMap(lambda x: x).collect()


# User was looking for rugs and interior paint & primer
# Use product recommender system to suggest similar products

["Mystic Fitz Roy Beige 9' 0 x 12' 0 Area Rug",
 'Florida Shag Beige/Multi 3 ft. x 5 ft. Floral Area Rug',
 '1 gal. #HDPG60 Misty Emerald Lake Flat Interior Paint and Primer',
 'Modern Gray/Multi 9 ft. x 12 ft. Vibrant Abstract Polyester Area Rug']

In [24]:
# filter only products in similar cluster
# exclude recently viewed products from recommendations
unique_clusters = filtered_data.select("cluster").distinct().rdd.flatMap(lambda x: x).collect()
possible_recommendations = clustered_data.filter(clustered_data['cluster'].isin(unique_clusters)).filter(~clustered_data['product_id'].isin(recently_viewed_products))

In [25]:
recommendations = possible_recommendations.groupby("cluster").agg(F.collect_list("product_id").alias("recommendations"))
recommendations_df = recommendations.toPandas()
recommendations_df['random_recommendations'] = recommendations_df['recommendations'].apply(lambda x: np.random.choice(x, size=5, replace=False).tolist())

In [26]:
recommendations_df.head()

,cluster,recommendations,random_recommendations
0,2,"[P2, P21, P52, P71, P87, P101, P108, P119, P12...","[P538, P2, P831, P1894, P1081]"
1,3,"[P6, P11, P16, P18, P24, P26, P30, P33, P40, P...","[P1729, P1771, P1970, P264, P1937]"


In [27]:
# write a python function to display the recommendations
def display_recommendations(row):
  # find the title of the product in df
  product_ids = row['random_recommendations']
  cluster = row.cluster

  titles = data. \
          filter(data["product_id"]. \
          isin(product_ids)).select("title").collect()

  print("\n")
  print("Recommendations for Cluster:", cluster)
  for title in titles:
    print(title[0])

recommendations_df.apply(display_recommendations, axis=1)



Recommendations for Cluster: 2
Large Tapestry Bolster Bed
Lyndhurst Multi/Green 10 ft. x 14 ft. Border Area Rug
Milas Cream 6 ft. x 9 ft. Oriental Polypropylene Area Rug
Distressed Beige / Cream 9 ft. x 12 ft. Rustic Damask Polypropylene Indoor/Outdoor Area Rug
Loom Modern Strie' Gray/Black 5 ft. x 8 ft. Area Rug


Recommendations for Cluster: 3
5 gal. #PR-W06 Prelude to Pink Flat Exterior Paint & Primer
5 gal. #PPU9-25 Eastern Bamboo Satin Enamel Exterior Paint & Primer
1 gal. #MQ4-31 Stardust Evening Extra Durable Satin Enamel Interior Paint & Primer
5 gal. #T17-03 Sepia Filter Flat Exterior Paint & Primer
8 oz. #510C-3 Rivers Edge Semi-Gloss Enamel Stain-Blocking Interior/Exterior Paint & Primer Sample


0    None
1    None
dtype: object